In [58]:
import sys
sys.path.append("..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, f1_score, balanced_accuracy_score, accuracy_score

from clean_data import feature_engineering_days

In [59]:
df = pd.read_csv("../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Cat"].copy()

In [60]:
df = feature_engineering_days(df)

df_cats = df[df["animal_type"] == "Cat"].copy()
df_cats = df_cats.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 7, 30, np.inf]
df_cats["target"] = pd.cut(
    df_cats["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [61]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter",
    "outcome_type", "outcome_subtype", "sex_upon_outcome",
    "intake_datetime", "animal_type", "young_and_cat", "datetime", "animal_id_intake"
]

X = df_cats.drop(columns=drop_cols, errors="ignore")
y = df_cats["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 20526 4399 4399


In [62]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0783285	test: 1.0806491	best: 1.0806491 (0)	total: 75.7ms	remaining: 1m 30s
100:	learn: 0.6616299	test: 0.8213291	best: 0.8213291 (100)	total: 9.89s	remaining: 1m 47s
200:	learn: 0.6290960	test: 0.8080689	best: 0.8080689 (200)	total: 17.8s	remaining: 1m 28s
300:	learn: 0.6140127	test: 0.8034968	best: 0.8034968 (300)	total: 25.6s	remaining: 1m 16s
400:	learn: 0.6017347	test: 0.8011330	best: 0.8011330 (400)	total: 33.4s	remaining: 1m 6s
500:	learn: 0.5917323	test: 0.7999037	best: 0.7998365 (474)	total: 42.4s	remaining: 59.1s
600:	learn: 0.5846394	test: 0.8000116	best: 0.7998115 (570)	total: 50.3s	remaining: 50.1s
700:	learn: 0.5773594	test: 0.7995467	best: 0.7993575 (675)	total: 58s	remaining: 41.3s
800:	learn: 0.5715768	test: 0.7990641	best: 0.7990626 (749)	total: 1m 5s	remaining: 32.8s
900:	learn: 0.5654259	test: 0.7992231	best: 0.7989848 (847)	total: 1m 13s	remaining: 24.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7989848387
bestIteration = 847


CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [63]:
X_train.columns

Index(['breed', 'color', 'found_location', 'intake_condition', 'intake_type',
       'name', 'sex', 'intact', 'is_mixed_breed', 'primary_breed',
       'is_mixed_color', 'primary_color', 'has_name', 'name_length',
       'age_days', 'day_of_week', 'month', 'year', 'is_weekend'],
      dtype='object')

# Evaluation


In [64]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.8032
Brier Score:       0.4722
Macro F1:          0.5714
Balanced Accuracy: 0.5695

Classification report:
              precision    recall  f1-score   support

           0       0.75      0.80      0.77      2343
           1       0.45      0.39      0.42      1165
           2       0.53      0.52      0.53       891

    accuracy                           0.64      4399
   macro avg       0.58      0.57      0.57      4399
weighted avg       0.62      0.64      0.63      4399



In [65]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >30 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >30): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >30 dni: 0.201
Oznaczone: 39.7% zwierząt
Precyzja: 0.400
Recall:   0.786
PR-AUC (klasa >30): 0.565


In [ ]:
def predict_dog(model, cat):
    probs = model.predict_proba(cat)[0]

    return pd.DataFrame({
        "Czas pobytu": ["≤7 dni", "7–30 dni", ">30 dni"],
        "Prawdopodobieństwo (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [67]:
cat = X_test.iloc[[5]]
print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_dog(model, cat))

9.135416666666666


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,95.42
1,7–30 dni,3.86
2,>30 dni,0.71


In [68]:
cat = X_test.iloc[[30]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])

display(predict_dog(model, cat))

9.902083333333334


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,68.01
1,7–30 dni,17.56
2,>30 dni,14.43


In [ ]:
long_stay = X_test[df_cats.loc[X_test.index, "time_in_shelter_days"] > 60]

cat = long_stay.iloc[[0]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_dog(model, cat))


67.85694444444445


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,15.41
1,7–30 dni,36.35
2,>30 dni,48.24


In [70]:
# import shap

# explainer = shap.TreeExplainer(model)
# X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
# shap_values = explainer(X_sample)

# print(shap_values.values.shape)

In [71]:
# class_idx = list(model.classes_).index(2)
# shap_long = shap_values[:, :, class_idx]
# shap.plots.beeswarm(shap_long, max_display=15)

In [ ]:
# cat = X_test.iloc[[0]]
# display(predict_dog(model, cat))

# dog_shap = explainer(cat)

# shap.plots.waterfall(
#      dog_shap[0, :, class_idx],
#      max_display=15
#  )